# OOP advanced
* in lesson 07 we already covered the basics of classes: `class`, `__init__`, `self`, dunder methods, properties and static/classmethods - if any of that sounds unfamiliar, go back there first
* in this lesson we build on top of that and go deeper: inheritance, multiple inheritance and MRO, polymorphism, abstract base classes, composition vs inheritance, mixins and dataclasses

## inheritance
* inheritance lets a class "inherit" attributes and methods from another class, so we dont have to repeat code
* the class we inherit from is called the "parent class" (or base/super class), the new class is the "child class" (or subclass)
* syntax: `class Child(Parent):` - the parentheses tell python which class to inherit from
* inside the child we can call `super().__init__(...)` to run the parent's constructor before adding our own stuff

In [1]:
# lets start with a simple parent class
class Animal:
    def __init__(self, name):
        self.name = name  # every animal has a name

    def speak(self):
        # a generic default implementation, subclasses can override this
        return f"{self.name} makes some generic animal sound"

    def describe(self):
        return f"{self.name} is an animal"


# Dog inherits everything from Animal, we just put Animal in the parentheses
class Dog(Animal):
    def __init__(self, name, breed):
        super().__init__(name)  # call the parent constructor so self.name gets set correctly
        self.breed = breed      # then add our own, dog-specific attribute

    # overriding: we completely replace the parent's speak() method with our own version
    def speak(self):
        return f"{self.name} barks: Woof!"

    # extending: we call the parent method first, then add MORE behaviour on top of it
    def describe(self):
        base_description = super().describe()  # get the parent's result first
        return f"{base_description} and specifically a {self.breed} dog"  # then extend it


my_dog = Dog("Rex", "Labrador")
print(my_dog.speak())      # overridden version is used
print(my_dog.describe())   # extended version, includes the parent's text too
print(isinstance(my_dog, Dog))     # True, obviously
print(isinstance(my_dog, Animal))  # True too! a Dog IS-A Animal, inheritance creates this relationship

Rex barks: Woof!
Rex is an animal and specifically a Labrador dog
True
True


## multiple inheritance and the Method Resolution Order (MRO)
* python allows a class to inherit from MORE than one parent class at the same time - unlike many other languages
* this gets tricky when several parents define the same method name - which one wins?
* python resolves this with a well defined algorithm called the "Method Resolution Order" (MRO), based on C3 linearization
* we can always inspect the MRO with `ClassName.__mro__` or `ClassName.mro()` to see the exact lookup order

In [2]:
# the classic "diamond inheritance" shape: two classes B and C both inherit from A, and D inherits from both B and C
class A:
    def who_am_i(self):
        return "A"

class B(A):
    def who_am_i(self):
        return "B"

class C(A):
    def who_am_i(self):
        return "C"

class D(B, C):  # note the order here: B before C
    pass  # D doesnt override who_am_i itself, so python needs to decide which parent's version to use


d = D()
print(d.who_am_i())  # "B" - because B comes first in the parentheses, and python looks left to right

# we can see the exact resolution order python computed for us
print(D.__mro__)
# or equivalently, as a list
print(D.mro())

# if we flip the inheritance order, the result changes too
class D2(C, B):  # now C comes first
    pass

print(D2().who_am_i())  # "C" this time
# expert note: the MRO is not JUST "left to right depth first", it also guarantees a parent never appears before its own child in the order
# thats the whole point of C3 linearization - it keeps the order consistent even in more complex diamond shapes

B
(<class '__main__.D'>, <class '__main__.B'>, <class '__main__.C'>, <class '__main__.A'>, <class 'object'>)
[<class '__main__.D'>, <class '__main__.B'>, <class '__main__.C'>, <class '__main__.A'>, <class 'object'>]
C


## polymorphism
* polymorphism means: the same method name behaves differently depending on which actual object it is called on
* we already saw a tiny bit of this with `speak()` above, lets make it more obvious by putting different subclasses into ONE list and calling the same method on all of them

In [3]:
class Cat(Animal):
    def speak(self):
        return f"{self.name} says: Meow!"


class Cow(Animal):
    def speak(self):
        return f"{self.name} says: Moo!"


# one list, mixed types - Dog, Cat, Cow all inherit from Animal, so they all fit into an "Animal" list conceptually
farm_animals = [Dog("Rex", "Labrador"), Cat("Whiskers"), Cow("Bessie"), Animal("Generic Thing")]

# we call the exact same method name on every single item, but the ACTUAL behaviour differs per real type
# this is polymorphism: python figures out at runtime which speak() implementation to use, based on the object's real class
for animal in farm_animals:
    print(animal.speak())

# this is super useful because the calling code doesnt need to know or care which exact subclass it is dealing with
# it just trusts that every Animal (whatever subclass) has a working speak() method

Rex barks: Woof!
Whiskers says: Meow!
Bessie says: Moo!
Generic Thing makes some generic animal sound


## abstract base classes (ABC)
* sometimes we want to FORCE subclasses to implement certain methods - a parent class that should never be instantiated on its own, only used as a blueprint
* python's `abc` module gives us `ABC` (a base class to inherit from) and the `@abstractmethod` decorator
* if a class still has unimplemented abstract methods, python refuses to instantiate it directly (raises `TypeError`) - subclasses MUST implement every abstract method or they cant be instantiated either

In [4]:
from abc import ABC, abstractmethod

class Shape(ABC):  # inheriting from ABC marks this class as abstract
    @abstractmethod
    def area(self):
        # no real implementation here, just a promise: "every subclass must provide this"
        pass

    def describe(self):
        # a NON-abstract, normal method is totally fine inside an ABC too, subclasses inherit it as-is
        return f"this shape has an area of {self.area()}"


# lets try to instantiate the abstract class directly - this must fail
try:
    generic_shape = Shape()
except TypeError as error:
    print("error:", error)  # cant instantiate abstract class Shape with abstract method area


class Square(Shape):
    def __init__(self, side):
        self.side = side
    # we implement the required area() method, so Square CAN be instantiated
    def area(self):
        return self.side ** 2


# an "incomplete" subclass that forgets to implement area()
class BrokenShape(Shape):
    pass  # oops, no area() implementation


square = Square(4)
print(square.describe())  # works fine, describe() uses our area() implementation

try:
    broken = BrokenShape()
except TypeError as error:
    print("error:", error)  # still fails, BrokenShape never implemented the abstract area() method

error: Can't instantiate abstract class Shape with abstract method area
this shape has an area of 16
error: Can't instantiate abstract class BrokenShape with abstract method area


## composition vs inheritance ("has-a" vs "is-a")
* inheritance models an "is-a" relationship (a Dog IS-A Animal)
* composition models a "has-a" relationship - one object simply HOLDS another object as an attribute, instead of inheriting from it
* rule of thumb often quoted in the industry: "favor composition over inheritance" - composition tends to be more flexible, avoids deep/fragile inheritance chains, and keeps classes more independent from each other

In [5]:
# composition example: a Car HAS an Engine, it does not inherit from Engine
class Engine:
    def __init__(self, horsepower):
        self.horsepower = horsepower

    def start(self):
        return f"engine with {self.horsepower}hp starting up, vroom!"


class Car:
    def __init__(self, brand, horsepower):
        self.brand = brand
        self.engine = Engine(horsepower)  # composition: Car HOLDS an Engine object as an attribute

    def start(self):
        # Car delegates the actual work to its engine, it doesnt need to know HOW an engine works internally
        return f"{self.brand}: {self.engine.start()}"


my_car = Car("Toyota", 150)
print(my_car.start())

# a flawed "is-a" alternative for comparison: making Car inherit from Engine would be semantically wrong
class FlawedCar(Engine):  # a Car IS NOT an Engine, this is a misuse of inheritance just to reuse code
    def __init__(self, brand, horsepower):
        super().__init__(horsepower)
        self.brand = brand

flawed_car = FlawedCar("Toyota", 150)
print(flawed_car.start())  # "works" technically, but a FlawedCar IS an Engine now (isinstance(flawed_car, Engine) is True)
print(isinstance(flawed_car, Engine))  # True - semantically weird, a car should not literally BE an engine
# composition avoids this problem entirely: my_car is NOT an Engine, it just happens to use one internally

Toyota: engine with 150hp starting up, vroom!
engine with 150hp starting up, vroom!
True


## mixins
* a mixin is a small class that isnt meant to stand on its own, it just adds one specific piece of reusable behaviour that gets "mixed in" via multiple inheritance
* mixins usually dont have their own `__init__`, they just provide extra methods that rely on attributes the "main" class already has (usually as a dict via `__dict__` or `vars()`)
* naming convention: mixin classes are often named with a `...Mixin` suffix

In [6]:
class SerializableMixin:
    # this mixin adds ONE useful method: turning any object's attributes into a plain dict
    def to_dict(self):
        return dict(self.__dict__)  # self.__dict__ holds every instance attribute as a dict, we just copy it


class Point(SerializableMixin):
    def __init__(self, x, y):
        self.x = x
        self.y = y


class User(SerializableMixin):
    def __init__(self, username, email):
        self.username = username
        self.email = email


# two completely unrelated classes (Point and User have nothing to do with each other) both get to_dict() for free
point = Point(3, 4)
user = User("alex", "alex@example.com")

print(point.to_dict())
print(user.to_dict())

# the mixin itself is never meant to be used standalone, it only makes sense combined with a "real" class
# this is a lightweight alternative to writing to_dict() again and again in every class that needs it

{'x': 3, 'y': 4}
{'username': 'alex', 'email': 'alex@example.com'}


## dataclasses
* writing `__init__`, `__repr__` and `__eq__` by hand for simple "data holder" classes gets repetitive fast
* the `@dataclass` decorator from the `dataclasses` module auto-generates all of that for us, based only on class-level type annotations
* default values work just like normal function parameter defaults - but a MUTABLE default (like a list) needs `field(default_factory=...)` instead of a plain default, otherwise every instance would share the SAME list object
* `frozen=True` makes instances immutable (like a tuple) - trying to change an attribute afterwards raises an error

In [7]:
# lets first write the "old fashioned" way by hand, so we can see what @dataclass saves us
class PersonManual:
    def __init__(self, name, age, hobbies=None):
        self.name = name
        self.age = age
        self.hobbies = hobbies if hobbies is not None else []  # manual workaround for the mutable default problem

    def __repr__(self):
        return f"PersonManual(name={self.name!r}, age={self.age!r}, hobbies={self.hobbies!r})"

    def __eq__(self, other):
        if not isinstance(other, PersonManual):
            return NotImplemented
        return (self.name, self.age, self.hobbies) == (other.name, other.age, other.hobbies)


manual_person = PersonManual("Alex", 30)
print(manual_person)
print(manual_person == PersonManual("Alex", 30))  # True, our manual __eq__ compares field by field


from dataclasses import dataclass, field

@dataclass
class Person:
    # just type-annotated class attributes - @dataclass reads these and builds __init__/__repr__/__eq__ automatically
    name: str
    age: int
    hobbies: list = field(default_factory=list)  # mutable default MUST use field(default_factory=...), never hobbies=[] directly


person = Person("Alex", 30)
print(person)                       # auto-generated __repr__, looks exactly like our manual one
print(person == Person("Alex", 30)) # auto-generated __eq__, True

# proof that default_factory really creates a NEW list per instance, not one shared list
person_a = Person("A", 1)
person_b = Person("B", 2)
person_a.hobbies.append("chess")
print(person_a.hobbies, person_b.hobbies)  # person_b.hobbies stays empty, each instance got its OWN list


# frozen=True makes the dataclass immutable
@dataclass(frozen=True)
class ImmutablePoint:
    x: int
    y: int

frozen_point = ImmutablePoint(1, 2)
try:
    frozen_point.x = 99  # this must fail, frozen dataclasses dont allow attribute reassignment
except Exception as error:
    print("error:", type(error).__name__, error)

PersonManual(name='Alex', age=30, hobbies=[])
True
Person(name='Alex', age=30, hobbies=[])
True
['chess'] []
error: FrozenInstanceError cannot assign to field 'x'


## quick comparison to other languages
* python allows multiple inheritance of classes directly, with an explicit, deterministic MRO algorithm (C3 linearization) resolving conflicts - Java and C# deliberately FORBID multiple class inheritance, they only allow implementing multiple *interfaces* instead
* C++ also allows multiple inheritance, but is notorious for the "diamond problem" and needs explicit "virtual inheritance" to resolve ambiguous shared base classes - python handles this automatically via its MRO, no extra syntax needed
* python's `abc.ABC` + `@abstractmethod` is the rough equivalent of Java's `interface`/`abstract class`, C#'s `interface`/`abstract class`, Rust's `trait`, or Go's implicitly satisfied structural interfaces (Go doesnt even need an explicit "implements" keyword)
* python's `@dataclass` is comparable to Java's `record` (since Java 14), Kotlin's `data class`, C#'s `record`, or Rust's `#[derive(Debug, PartialEq)] struct` - all of these auto-generate the boilerplate (constructor, equality, string representation) for simple data-holding types

## Exercises
1. write an abstract base class `Shape` (using `ABC` and `@abstractmethod`) with an abstract method `area()`, then write two concrete subclasses `Circle` (constructor takes `radius`) and `Rectangle` (constructor takes `width` and `height`) that each implement `area()` correctly
2. fix the following diamond-inheritance MRO puzzle: given `class Base: def greet(self): return "base"`, `class Left(Base): def greet(self): return "left"`, `class Right(Base): def greet(self): return "right"`, define `class Bottom(Left, Right): pass` and predict/print what `Bottom().greet()` returns and print `Bottom.__mro__` to confirm why
3. take this hand-written class and rewrite it as an equivalent `@dataclass`: `class Book:` with `__init__(self, title, author, tags=None)` (careful with the mutable default!), plus a working `__repr__` and `__eq__`
4. write a mixin class `ToDictMixin` (or reuse the idea from the lesson) that adds a `to_dict()` method, then mix it into two unrelated classes of your choice (e.g. `Product` and `Employee`) and prove both get a working `to_dict()`

In [8]:
# TODO: solve exercise 1 here (abstract Shape with Circle and Rectangle)


# TODO: solve exercise 2 here (diamond inheritance MRO puzzle)


# TODO: solve exercise 3 here (Book as a dataclass)


# TODO: solve exercise 4 here (ToDictMixin mixed into two unrelated classes)

### sample solutions
* try the exercises yourself first, only look here if you are stuck

In [9]:
# sample solution 1 - abstract Shape with Circle and Rectangle
from abc import ABC, abstractmethod
import math

class Shape(ABC):
    @abstractmethod
    def area(self):
        pass

class Circle(Shape):
    def __init__(self, radius):
        self.radius = radius
    def area(self):
        return math.pi * self.radius ** 2

class Rectangle(Shape):
    def __init__(self, width, height):
        self.width = width
        self.height = height
    def area(self):
        return self.width * self.height

print(round(Circle(2).area(), 2))
print(Rectangle(3, 4).area())


# sample solution 2 - diamond inheritance MRO puzzle
class Base:
    def greet(self):
        return "base"

class Left(Base):
    def greet(self):
        return "left"

class Right(Base):
    def greet(self):
        return "right"

class Bottom(Left, Right):
    pass

print(Bottom().greet())  # "left" - Left comes first in Bottom(Left, Right)
print(Bottom.__mro__)    # shows the full lookup order: Bottom -> Left -> Right -> Base -> object


# sample solution 3 - Book as a dataclass
from dataclasses import dataclass, field

@dataclass
class Book:
    title: str
    author: str
    tags: list = field(default_factory=list)  # mutable default needs default_factory

book_one = Book("Dune", "Frank Herbert")
book_two = Book("Dune", "Frank Herbert")
print(book_one)               # auto __repr__
print(book_one == book_two)   # auto __eq__, True since all fields match


# sample solution 4 - ToDictMixin mixed into two unrelated classes
class ToDictMixin:
    def to_dict(self):
        return dict(self.__dict__)

class Product(ToDictMixin):
    def __init__(self, name, price):
        self.name = name
        self.price = price

class Employee(ToDictMixin):
    def __init__(self, name, salary):
        self.name = name
        self.salary = salary

print(Product("Laptop", 999).to_dict())
print(Employee("Alex", 5000).to_dict())

12.57
12
left
(<class '__main__.Bottom'>, <class '__main__.Left'>, <class '__main__.Right'>, <class '__main__.Base'>, <class 'object'>)
Book(title='Dune', author='Frank Herbert', tags=[])
True
{'name': 'Laptop', 'price': 999}
{'name': 'Alex', 'salary': 5000}


* congratulation you finished this lesson

## what we learned
* inheritance: `class Child(Parent):`, calling `super().__init__(...)`, overriding vs extending a parent method
* multiple inheritance and the Method Resolution Order (MRO), inspecting it with `__mro__`/`mro()`
* polymorphism: calling the same method name on a mixed list of subclass instances, each behaving differently
* abstract base classes with `abc.ABC` and `@abstractmethod`, and why an incomplete subclass still cant be instantiated
* composition vs inheritance ("has-a" vs "is-a") and why "favor composition over inheritance" is common advice
* mixins: small reusable classes mixed in via multiple inheritance to add one specific piece of behaviour
* `@dataclass`: auto-generated `__init__`/`__repr__`/`__eq__`, `field(default_factory=...)` for mutable defaults, `frozen=True` for immutability